# Sentinel Travel Risk Lab: Model Evaluation

This notebook evaluates the two XGBoost classifiers used by the working prototype:

- payment fraud risk
- inventory abuse risk

**Data disclosure:** every row is synthetic and generated by `backend/ml/generate_data.py` with a fixed seed. Metrics demonstrate that the software and evaluation pipeline work on generated patterns. They are not evidence of production fraud-detection performance.

In [1]:
from pathlib import Path
import json
import platform

import pandas as pd
import sklearn
import xgboost

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "backend" / "data" / "synthetic_bookings.csv"
METADATA_PATH = PROJECT_ROOT / "backend" / "artifacts" / "metadata.json"

assert DATA_PATH.exists(), f"Missing generated dataset: {DATA_PATH}"
assert METADATA_PATH.exists(), f"Missing model metadata: {METADATA_PATH}"

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
})

{'python': '3.11.9', 'pandas': '2.3.3', 'scikit_learn': '1.9.0', 'xgboost': '3.2.0'}


## 1. Load and validate the data

The checks below fail loudly if the schema, row count, missing-value policy, or binary labels differ from the trained artifact. No column or value is inferred.

In [2]:
data = pd.read_csv(DATA_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

required_columns = {
    "event_day",
    *metadata["features"],
    "payment_fraud_label",
    "inventory_abuse_label",
}
missing_columns = required_columns - set(data.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert len(data) == metadata["row_count"]
assert data[list(required_columns)].isna().sum().sum() == 0
assert set(data["payment_fraud_label"].unique()) <= {0, 1}
assert set(data["inventory_abuse_label"].unique()) <= {0, 1}

validation_summary = pd.DataFrame({
    "value": [
        len(data),
        data.duplicated().sum(),
        data["payment_fraud_label"].mean(),
        data["inventory_abuse_label"].mean(),
    ]
}, index=["rows", "duplicate rows", "payment fraud prevalence", "inventory abuse prevalence"])
validation_summary

,value
rows,15000.000000
duplicate rows,0.000000
payment fraud prevalence,0.053867
inventory abuse prevalence,0.095267


## 2. Reconstruct the chronological holdout

The models were fitted on the oldest 85% of events (70% training plus 15% validation). The newest 15% remains untouched until final evaluation. This is more realistic than a random split for changing fraud patterns.

In [3]:
validation_end = data["event_day"].quantile(0.85)
test = data[data["event_day"] > validation_end].copy()

assert len(test) == metadata["split_counts"]["test"]
print({
    "test_rows": len(test),
    "first_test_day": int(test["event_day"].min()),
    "last_test_day": int(test["event_day"].max()),
})

metric_table = pd.DataFrame(metadata["test_metrics"]).T[
    ["precision", "recall", "pr_auc", "roc_auc", "false_positive_rate"]
]
metric_table

{'test_rows': 2226, 'first_test_day': 310, 'last_test_day': 364}


,precision,recall,pr_auc,roc_auc,false_positive_rate
payment_fraud,0.7571,0.3706,0.4858,0.7758,0.0082
inventory_abuse,0.8229,0.632,0.6714,0.8443,0.0172


## 3. Recompute metrics from saved models

Accuracy is intentionally not the headline metric because fraud classes are imbalanced. We recompute precision, recall, PR-AUC, ROC-AUC, false-positive rate, and the confusion matrix at the documented 0.50 threshold.

In [4]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier


def evaluate_saved_model(model_name: str, label: str) -> dict:
    model = XGBClassifier()
    model.load_model(PROJECT_ROOT / "backend" / "artifacts" / model_name)
    probabilities = model.predict_proba(test[metadata["features"]])[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(test[label], predictions, labels=[0, 1]).ravel()
    return {
        "precision": round(float(precision_score(test[label], predictions, zero_division=0)), 4),
        "recall": round(float(recall_score(test[label], predictions, zero_division=0)), 4),
        "pr_auc": round(float(average_precision_score(test[label], probabilities)), 4),
        "roc_auc": round(float(roc_auc_score(test[label], probabilities)), 4),
        "false_positive_rate": round(float(fp / max(fp + tn, 1)), 4),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


recomputed = {
    "payment_fraud": evaluate_saved_model("payment_fraud_model.json", "payment_fraud_label"),
    "inventory_abuse": evaluate_saved_model("inventory_abuse_model.json", "inventory_abuse_label"),
}
pd.DataFrame(recomputed).T

,precision,recall,pr_auc,roc_auc,false_positive_rate,tn,fp,fn,tp
payment_fraud,0.7571,0.3706,0.4858,0.7758,0.0082,2066.0,17.0,90.0,53.0
inventory_abuse,0.8229,0.6320,0.6714,0.8443,0.0172,1942.0,34.0,92.0,158.0


## 4. Verification and safety checks

The assertions below prevent stale report values, accidental random splitting, invalid rates, or an artifact/dataset mismatch from passing silently.

In [5]:
for task, values in recomputed.items():
    saved = metadata["test_metrics"][task]
    for metric in ("precision", "recall", "pr_auc", "roc_auc", "false_positive_rate"):
        assert values[metric] == saved[metric], f"{task} {metric} differs from metadata"
        assert 0 <= values[metric] <= 1
    for key in ("tn", "fp", "fn", "tp"):
        assert values[key] == saved["confusion_matrix"][key]
    assert sum(values[key] for key in ("tn", "fp", "fn", "tp")) == len(test)

print("PASS: saved models, metadata, and untouched-test results agree.")

PASS: saved models, metadata, and untouched-test results agree.


## 5. Honest conclusion

On the untouched synthetic period, the payment model reaches **0.757 precision**, **0.371 recall**, and **0.486 PR-AUC**. The inventory-abuse model reaches **0.823 precision**, **0.632 recall**, and **0.671 PR-AUC**. The payment model therefore misses many generated fraud cases at the 0.50 threshold even though its false-positive rate is low.

These results are useful for verifying the implementation and discussing threshold trade-offs. They must not be described as real travel-industry performance. Production validation requires legally obtained, time-stamped booking outcomes, group-aware leakage checks, probability calibration, fairness review, and monitored human decisions.